# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Genera el mapa interactivo, rankings de cadenas, provincias y barrios CABA.
**No requiere cargar ningún archivo — todo se descarga automáticamente.**

▶️ **Ejecutar todo** (`Runtime → Run all`) y esperar ~10 minutos.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — única celda modificable
# ══════════════════════════════════════════════════════════════════

# Mes a analizar (formato 'YYYY-MM').
# Dejalo en None para usar el último mes disponible automáticamente.
MES = None          # ← cambiá a '2026-04' si querés un mes específico

# ── No modificar nada de acá para abajo ───────────────────────────
REPO_URL      = 'https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git'
MANIFEST_URL  = 'https://raw.githubusercontent.com/santiagoriverti/analisis_precios_minoristas_UADE/main/data/drive_manifest.json'
DIR_DATOS     = '/content/datos_sepa'
DIR_REPO      = '/content/repo'

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 1 — Instalación y clonado del repositorio
# ══════════════════════════════════════════════════════════════════
import os, sys

if not os.path.exists(DIR_REPO):
    os.system(f'git clone -q {REPO_URL} {DIR_REPO}')
    print('✓ Repositorio clonado')
else:
    os.system(f'git -C {DIR_REPO} pull -q')
    print('✓ Repositorio actualizado')

os.system('pip install -q gdown folium branca openpyxl pyarrow')
sys.path.insert(0, f'{DIR_REPO}/src')
os.makedirs(DIR_DATOS, exist_ok=True)
print('✓ Dependencias instaladas')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 2 — Cargar manifest y elegir el mes a analizar
# ══════════════════════════════════════════════════════════════════
import requests, json

manifest = requests.get(MANIFEST_URL).json()

# Filtrar solo meses con IDs reales (no 'COMPLETAR')
meses_disponibles = sorted(
    m for m, v in manifest['meses'].items()
    if isinstance(v, dict)
    and v.get('parte1_id', 'COMPLETAR') != 'COMPLETAR'
    and v.get('parte2_id', 'COMPLETAR') != 'COMPLETAR'
    and not m.startswith('_')
)

if not meses_disponibles:
    raise RuntimeError(
        'El manifest no tiene meses cargados todavía.\n'
        'Correr scripts/actualizar_manifest.py para cargar los IDs de los archivos.'
    )

# Elegir mes
MES_ANALIZAR = MES if MES and MES in manifest['meses'] else meses_disponibles[-1]

print(f'Meses disponibles en el manifest: {meses_disponibles}')
print(f'Mes seleccionado: {MES_ANALIZAR}')

if MES and MES not in manifest['meses']:
    print(f'⚠ El mes {MES} no está en el manifest — usando el último disponible ({MES_ANALIZAR})')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 3 — Descargar archivos desde Google Drive (sin autenticación)
# ══════════════════════════════════════════════════════════════════
import gdown
from pathlib import Path

datos_mes = manifest['meses'][MES_ANALIZAR]
datos_maestros = manifest['maestros']

MES_TAG = MES_ANALIZAR.replace('-', '')   # '2026-04' → '202604'
MES_MM  = MES_ANALIZAR[5:]               # '2026-04' → '04'
MES_AAAA = MES_ANALIZAR[:4]              # '2026-04' → '2026'
MES_MMAAAA = f'{MES_MM}{MES_AAAA}'      # '042026'

archivos_a_descargar = [
    (datos_mes['parte1_id'],      f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte1COMPLETO.csv.gz',  f'SEPA parte 1'),
    (datos_mes['parte2_id'],      f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte2COMPLETO.csv.gz',  f'SEPA parte 2'),
    (datos_maestros['productos_id'],  f'{DIR_DATOS}/Maestro de Productos Interno.xlsx',    'Maestro productos'),
    (datos_maestros['sucursales_id'], f'{DIR_DATOS}/maestro_sucursales_completo.xlsx',      'Maestro sucursales'),
]

for file_id, output_path, label in archivos_a_descargar:
    p = Path(output_path)
    if p.exists() and p.stat().st_size > 100_000:
        print(f'  ↩ {label} ya descargado ({p.stat().st_size/1e6:.0f} MB)')
        continue
    print(f'  ↓ Descargando {label}...')
    gdown.download(
        id=file_id,
        output=output_path,
        quiet=False,
        fuzzy=True,
    )
    print(f'  ✓ {label}: {Path(output_path).stat().st_size/1e6:.0f} MB')

print(f'\n✓ Todos los archivos listos en {DIR_DATOS}/')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 4 — Cargar y limpiar datos
# ══════════════════════════════════════════════════════════════════
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

print(f'Cargando precios de {MES_ANALIZAR}...')
frames = []
for chunk in iter_semester_csvgz(Path(DIR_DATOS), ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('Sin datos. Verificá que los archivos se descargaron correctamente.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} registros | {df_long["ean_norm"].nunique()} EANs de canasta')
print(f'  Período: {df_long["fecha"].min().date()} → {df_long["fecha"].max().date()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 5 — Enriquecer con maestro de sucursales
# ══════════════════════════════════════════════════════════════════
mb = load_master_branches(f'{DIR_DATOS}/maestro_sucursales_completo.xlsx')
df_enriched = enrich_with_branches(df_long, mb)
del df_long; gc.collect()

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
n_suc = len(df_enriched[branch_cols].drop_duplicates())

print(f'✓ {n_suc:,} sucursales | {df_enriched.get("provincia", pd.Series()).nunique()} provincias | {df_enriched.get("cadena", pd.Series()).nunique()} cadenas')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 6 — Calcular canasta mensual por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)

df_mes = df_branch[df_branch['mes'] == MES_ANALIZAR].copy()
if df_mes.empty:
    raise RuntimeError(f'Sin datos para {MES_ANALIZAR}. Meses en cache: {sorted(df_branch["mes"].unique())}')

avg = df_mes['canasta_total'].mean()
cv  = df_mes['canasta_total'].std() / avg * 100

print(f'\n══ ICM-UADE — {MES_ANALIZAR} ══')
print(f'Sucursales:       {len(df_mes):>8,}')
print(f'Canasta promedio: ${avg:>12,.0f}'.replace(',', '.'))
print(f'Canasta mediana:  ${df_mes["canasta_total"].median():>12,.0f}'.replace(',', '.'))
print(f'Dispersión (CV):  {cv:>11.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 7 — Mapa interactivo por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.viz.maps import make_branch_map
from IPython.display import IFrame

suc_cols = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                       'sucursales_nombre','localidad_nombre','barrio_caba']
                           if c in df_enriched.columns]
df_suc = df_enriched[suc_cols].drop_duplicates(subset=branch_cols)

mapa_path = f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html'
make_branch_map(df_mes, df_suc, mapa_path, mes=MES_ANALIZAR)
print(f'✓ Mapa generado ({Path(mapa_path).stat().st_size/1e6:.0f} MB) — visualizando abajo:')

IFrame(mapa_path, width='100%', height=600)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 8 — Rankings de cadenas (nacional y AMBA)
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_mes, df_suc, mes=MES_ANALIZAR)
rank_amba = amba_ranking(df_mes, df_suc, mes=MES_ANALIZAR)

print('── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .style.format({'canasta_promedio': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'}).hide(axis='index'))

png_nac = f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png'
plot_chain_ranking(rank_nac, png_nac, title=f'Ranking Nacional — {MES_ANALIZAR}')
ipy_display(Image(png_nac))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .style.format({'canasta_promedio': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'}).hide(axis='index'))
    png_amba = f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png'
    plot_chain_ranking(rank_amba, png_amba, title=f'Ranking AMBA — {MES_ANALIZAR}')
    ipy_display(Image(png_amba))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 9 — Ranking provincial
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import aggregate_by_province
from sepa.analysis.basket import basket_by_province
from sepa.viz.charts import plot_province_ranking

if 'provincia' in df_suc.columns:
    df_prov = aggregate_by_province(df_mes, df_suc)
    df_rank_prov = basket_by_province(df_prov, MES_ANALIZAR)

    print('── Ranking Provincial ──')
    display(df_rank_prov[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .style.format({'canasta_provincia': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'}).hide(axis='index'))

    png_prov = f'/content/ranking_provincias_{MES_MMAAAA}.png'
    plot_province_ranking(df_prov, png_prov, mes=MES_ANALIZAR)
    ipy_display(Image(png_prov))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 10 — Ranking barrios CABA
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.basket import barrio_ranking_caba

df_barrios = barrio_ranking_caba(df_mes, df_suc, mes=MES_ANALIZAR)
if not df_barrios.empty:
    print(f'── Ranking Barrios CABA ({len(df_barrios)} barrios) ──')
    display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
            .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta','vs_promedio_caba_pct':'vs CABA %'})
            .style.format({'canasta': '${:,.0f}', 'vs CABA %': '{:+.1f}%'}).hide(axis='index'))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 11 — Descargar outputs a tu computadora
# ══════════════════════════════════════════════════════════════════
from google.colab import files
from pathlib import Path

outputs = [
    (f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html', 'mapa (HTML interactivo)'),
    (f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png',   'ranking nacional (PNG)'),
    (f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png',       'ranking AMBA (PNG)'),
    (f'/content/ranking_provincias_{MES_MMAAAA}.png',          'ranking provincial (PNG)'),
]

print('Descargando archivos a tu computadora...')
for ruta, label in outputs:
    if Path(ruta).exists():
        files.download(ruta)
        print(f'  ✓ {label}')
    else:
        print(f'  ⚠ No generado: {label}')

print(f'\n✓ Análisis completo de {MES_ANALIZAR} finalizado.')